# Notebook 2.1  From letters to sounds: a small Arabic Grapheme-to-Phoneme converter

**Companion to Chapter 2, *Introduction to Arabic Speech Technologies*.**

**Goal.** Turn diacritized Modern Standard Arabic text into a phoneme sequence with a small
rule-based Grapheme-to-Phoneme (G2P) converter that handles the emphatic consonants and
gemination (Shadda), evaluate it against a hand-labeled set, and *see* the acoustics of
emphasis: the lower second formant (F2) in a vowel next to an emphatic consonant (Section 2.4).

Runs with no downloads (the audio is synthesized). Optional clips: [Common Voice Arabic]
(https://commonvoice.mozilla.org/ar) (state the release version and license you use).

## 1. A rule-based G2P for diacritized MSA

In [ ]:
# Map each Arabic consonant to one phone symbol (Q marks emphatic/pharyngeal).
CONS = {
    'ب':'b','ت':'t','ث':'th','ج':'dZ','ح':'H','خ':'x','د':'d','ذ':'dh','ر':'r','ز':'z',
    'س':'s','ش':'S','ص':'sQ','ض':'dQ','ط':'tQ','ظ':'DQ','ع':'?Q','غ':'G','ف':'f','ق':'q',
    'ك':'k','ل':'l','م':'m','ن':'n','ه':'h','و':'w','ي':'j','ء':'?','أ':'?','إ':'?',
    'ة':'t','ى':'aa',
}
FATHA, DAMMA, KASRA = '\u064E', '\u064F', '\u0650'
SUKUN, SHADDA = '\u0652', '\u0651'
FATHATAN, DAMMATAN, KASRATAN = '\u064B', '\u064C', '\u064D'
ALEF = 'ا'
HARAKA = {FATHA:'a', DAMMA:'u', KASRA:'i'}
TANWEEN = {FATHATAN:('a','n'), DAMMATAN:('u','n'), KASRATAN:('i','n')}
MARKS = set(HARAKA) | {SUKUN, SHADDA} | set(TANWEEN)

def g2p(word, pronounce_tanween=False):
    """Rule-based G2P for diacritized MSA. Handles emphatics, gemination (Shadda) in any
    mark order, long vowels, and (optionally) Tanween. Tanween defaults to the pausal
    pronunciation (dropped), as is common for ASR lexicons."""
    phones = []; i = 0; n = len(word)
    while i < n:
        ch = word[i]
        if ch in CONS:
            j = i + 1; following = []
            while j < n and word[j] in MARKS:
                following.append(word[j]); j += 1
            base = CONS[ch]
            phones.append(base)
            if SHADDA in following:
                phones.append(base)            # gemination doubles the consonant
            haraka = [m for m in following if m in HARAKA]
            if haraka:
                v = HARAKA[haraka[0]]
                nb = word[j] if j < n else ''  # long vowel = haraka + matching letter
                if   v=='a' and nb==ALEF: phones.append('aa'); j += 1
                elif v=='u' and nb=='و':  phones.append('uu'); j += 1
                elif v=='i' and nb=='ي':  phones.append('ii'); j += 1
                else: phones.append(v)
            tanw = [m for m in following if m in TANWEEN]
            if tanw and pronounce_tanween:
                phones.extend(TANWEEN[tanw[0]])
            i = j; continue
        if ch == ALEF:
            phones.append('a' if not phones else 'aa')  # initial hamzat-wasl 'a', else long
        i += 1
    return phones

tests = ['كَتَبَ', 'طِينٌ', 'تِينٌ', 'مُدَرِّسٌ', 'صَيْفٌ']
for w in tests:
    print(w, '->', ' '.join(g2p(w)))

## 2. Evaluate against a hand-labeled set (phone error rate)

In [ ]:
def _levenshtein(ref, hyp):
    """Token-level edit distance with operation counts (S, D, I)."""
    n, m = len(ref), len(hyp)
    d = [[0]*(m+1) for _ in range(n+1)]
    for i in range(n+1): d[i][0] = i
    for j in range(m+1): d[0][j] = j
    for i in range(1, n+1):
        for j in range(1, m+1):
            cost = 0 if ref[i-1] == hyp[j-1] else 1
            d[i][j] = min(d[i-1][j]+1, d[i][j-1]+1, d[i-1][j-1]+cost)
    return d[n][m]

def error_rate(ref_tokens, hyp_tokens):
    if len(ref_tokens) == 0:
        return 0.0 if len(hyp_tokens) == 0 else 1.0
    return _levenshtein(ref_tokens, hyp_tokens) / len(ref_tokens)

def wer(ref, hyp):
    return error_rate(ref.split(), hyp.split())

def cer(ref, hyp):
    return error_rate(list(ref.replace(" ", "")), list(hyp.replace(" ", "")))

# (diacritized word, gold phone sequence). Tanween is dropped (pausal forms).
# The last item, الشمس 'the sun', is a sun-letter word: the article's laam assimilates
# to the following ش, which a simple rule-based G2P does not model, so it is a deliberate
# error case that motivates richer G2P.
gold = [
    ('كَتَبَ',    ['k','a','t','a','b','a']),
    ('طِينٌ',     ['tQ','ii','n']),
    ('تِينٌ',     ['t','ii','n']),
    ('مُدَرِّسٌ',  ['m','u','d','a','r','r','i','s']),
    ('الشَّمْس',   ['a','S','S','a','m','s']),
]
tot_err = tot_len = 0
for w, ref in gold:
    hyp = g2p(w)
    e = _levenshtein(ref, hyp)
    tot_err += e; tot_len += len(ref)
    flag = '   <- sun-letter assimilation missed' if e else ''
    print(f'{w:10} gold={ref}  hyp={hyp}  errors={e}{flag}')
print()
print('Phone Error Rate: {:.1f}%'.format(100*tot_err/tot_len))
print('The only error is the sun-letter (laam) assimilation in الشمس, a known limitation',
      'of simple rule-based G2P.')

## 3. See the emphatic effect: F2 lowering

We synthesize two vowels with simple formant resonators. The emphatic context (as next to
**ط** /tQ/ in طين *ṭīn* 'mud') lowers the second formant of the vowel; the plain context
(next to **ت** /t/ in تين *tīn* 'figs') keeps it high. Plot titles use transliteration so the
figure renders cleanly everywhere.

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.signal import spectrogram, lfilter

sr = 16000; dur = 0.5
t = np.linspace(0, dur, int(sr*dur), endpoint=False)

def synth_vowel(f0, formants, sr, t):
    # glottal source: impulse train at f0
    src = np.zeros_like(t)
    period = int(sr/f0)
    src[::period] = 1.0
    sig = np.zeros_like(t)
    for fF, bw in formants:
        r = np.exp(-np.pi*bw/sr)
        theta = 2*np.pi*fF/sr
        a = [1, -2*r*np.cos(theta), r*r]
        sig = sig + lfilter([1], a, src)
    sig = sig/np.max(np.abs(sig))
    return sig

# /ii/-like vowel: F1 low. Emphatic lowers F2 (~1600) vs plain high F2 (~2300).
plain    = synth_vowel(120, [(300,60),(2300,90),(3000,120)], sr, t)
emphatic = synth_vowel(120, [(350,60),(1600,90),(2700,120)], sr, t)

fig, ax = plt.subplots(1, 2, figsize=(11,4), sharey=True)
for a, (sig, name, f2) in zip(ax, [(plain,'tiin (figs), plain /t/',2300),
                                   (emphatic,'tiin -> ṭiin (mud), emphatic /tQ/',1600)]):
    f, tt, Sxx = spectrogram(sig, sr, nperseg=256, noverlap=192)
    a.pcolormesh(tt, f, 10*np.log10(Sxx+1e-10), shading='gouraud')
    a.axhline(f2, color='w', ls='--', lw=1)
    a.set_title(name + f'  (F2 ~ {f2} Hz)')
    a.set_xlabel('time (s)'); a.set_ylim(0, 4000)
ax[0].set_ylabel('frequency (Hz)')
plt.tight_layout(); plt.savefig('ch02_emphatic_F2.png', dpi=110)
print('saved ch02_emphatic_F2.png; note the lower dashed F2 line in the emphatic panel')

## 4. What to notice

The dashed line marks the second formant. In the emphatic panel it sits lower, the
coarticulatory backing described in Section 2.4. Measure F2 in the steady middle of the vowel,
not in the consonant. With real Common Voice clips of طين/تين you can repeat the measurement
on natural speech.

## Exercise solutions

Chapter 2's exercises are mostly phonetic reasoning; the ones that map to code use the `g2p` converter and the synthesis from earlier cells.

**Exercise (G2P and the sun-letter limitation).** Run the converter on a few words and show the one case it misses, the sun-letter assimilation in الشمس, which motivates a richer analyzer.

In [ ]:
for w in ['كَتَبَ','طِينٌ','تِينٌ','مُدَرِّسٌ','الشَّمْس']:
    print(w, '->', ' '.join(g2p(w)))
print('note: الشمس should assimilate the article laam to the sun letter (a-sh-sh-...),')
print('which a simple rule-based converter does not model.')

**Exercise (emphatic F2).** Use a wideband view to locate the vowel, then measure F2 at the midpoint; expect a lower F2 next to the emphatic ط than next to the plain ت. **IPA / source-filter / diacritization / clitic exercises (conceptual):** answers are in the chapter Solutions; the clitic case وبكتابهم segments as wa + bi + kitāb + hum.